# EXIT retrieval: public Kaggle Dataset checkpoints
Check the Kaggle API before heavy work. Prepare in /tmp, then upload useful processed checkpoints every 120 minutes and after SQLite/sample/train/build completion. Download/extraction progress stays local. All Datasets, including the tiny API-check Dataset, are public.

In [ ]:
from pathlib import Path
import os
REPO = Path('/kaggle/working/EXIT')  # adjust checkout path
os.chdir(REPO)

In [ ]:
%pip install -r requirements-retrieval.txt

## Configure
This cell writes a notebook-local config copy, without changing the repo template. Existing state in /tmp/retrieval-state is reused. Use a new Dataset prefix when changing PQ settings. All generated Datasets are public.

In [ ]:
import json
config = json.loads(Path('configs/retrieval_kaggle.json').read_text())
config['owner'] = 'YOUR_USERNAME'  # REQUIRED: your actual Kaggle username
config['dataset_prefix'] = 'exit-wikipedia-retrieval'
config['checkpoint_minutes'] = 120
config['time_budget_minutes'] = 600
# Optional attached input sources:
# config['shards'] = '/kaggle/input/embedding-source/passages_*'
# config['tsv'] = '/kaggle/input/wiki-corpus/psgs_w100.tsv'
# Optional checkpoint/corpus attached via Add Input:
# config['checkpoint_input'] = '/kaggle/input/exit-wikipedia-retrieval-checkpoint'
# config['corpus_input'] = '/kaggle/input/exit-wikipedia-retrieval-corpus'
assert config['owner'] != 'YOUR_USERNAME', 'Set your Kaggle username first'
CONFIG = '/kaggle/working/retrieval-session.json'
Path(CONFIG).write_text(json.dumps(config, indent=2))
print(f'Saved config: {CONFIG}')

## Authentication
Create and enable a Kaggle Secret named KAGGLE_API_TOKEN. It is passed only through the environment; it is not included in snapshots. For a legacy key, use the commented alternative.
Use a personal API token with Dataset read/write permissions. The uploader prioritizes this explicit token over the built-in notebook token.

In [ ]:
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ['KAGGLE_API_TOKEN'] = secrets.get_secret('KAGGLE_API_TOKEN')
# Legacy alternative (replace the token line above):
# os.environ['KAGGLE_USERNAME'] = secrets.get_secret('KAGGLE_USERNAME')
# os.environ['KAGGLE_KEY'] = secrets.get_secret('KAGGLE_KEY')

## Check Kaggle API before heavy work
Creates/updates <prefix>-api-check with a tiny random probe, waits for processing, then downloads and verifies it. This is an API test, not a corpus/index checkpoint. Stop if this cell fails; fix Secrets/owner/permissions before resume. The probe Dataset is public and is not deleted automatically.

In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, '-u', '-m', 'retrieval.prepare', 'check-api', '--config', CONFIG], check=True)

In [ ]:
!python -u -m retrieval.prepare preflight --work /tmp/retrieval-state

## Restore only in a new session
Leave commented for a first run or when continuing with intact local state. If enabled, restore downloads the latest checkpoint and exact referenced corpus version (or reads the configured attached Inputs). Conflicting local files are not overwritten.

In [ ]:
# subprocess.run([sys.executable, '-u', '-m', 'retrieval.prepare', 'restore', '--config', CONFIG], check=True)

## Check status
Read-only. Shows completed/current steps, counts, the next step and confirmed remote Dataset versions. Local progress is separate from remotely saved progress.

In [ ]:
!python -u -m retrieval.prepare status --config "{CONFIG}"

## Resume automatically
Each new resume process repeats the small API upload/read-back check before any download or processing. CHECKPOINT UPLOAD logs are separate from API CHECK logs. No empty download-only checkpoints are uploaded; unchanged processed payload is skipped. Leave time for final uploads before the session limit.
START/DONE identify each long operation. WORKING heartbeats report elapsed time every 30 seconds; PROGRESS shows counters when available. A heartbeat does not guarantee that a blocking external service is making progress.

In [ ]:
subprocess.run([sys.executable, '-u', '-m', 'retrieval.prepare', 'resume', '--config', CONFIG], check=True)

In [ ]:
!python -u -m retrieval.prepare status --config "{CONFIG}"

## Optional: save durable state immediately
Run only after the active resume command has stopped. The command uploads only useful changed processed data. If there is only download/extraction state, or nothing has changed, it prints CHECKPOINT SKIP.

In [ ]:
# !python -u -m retrieval.prepare checkpoint --config "{CONFIG}"

## Next session / inference
New preparation session: load config and Secrets, enable restore, then run status and resume. Finished: attach the public Dataset <owner>/<prefix>-artifact to the inference notebook. Its four artifact files can be used directly; pack/restore packages are unnecessary for the new workflow. Raw input downloads are not checkpointed remotely and may need downloading again.